# Perbandingan Model Tambahan: LinearSVC, CatBoost, BalancedRandomForest, Logistic Regression
## Ekstensi Notebook Utama Prediksi Penangkapan Kejahatan LAPD

Notebook ini adalah **ekstensi terpisah** dari notebook utama LightGBM + XGBoost. Tujuannya: menguji 4 model tambahan pada **dataset dan preprocessing yang identik** agar perbandingan fair.

### Hipotesis
Model gradient boosting (LightGBM) kemungkinan masih unggul karena:
- Data tabular dengan fitur campuran (numerik + kategorikal) → keunggulan tree-based model
- Class imbalance 9:91 gradient boosting lebih adaptif via `class_weight`

Namun, 4 model berikut diuji sebagai **challenger**:

| Model | Karakteristik | Handling Imbalance |
|---|---|---|
| **Logistic Regression** | Linear baseline yang kuat; interpretabel; butuh scaling | `class_weight='balanced'` |
| **LinearSVC** | SVM linear; efisien di data besar; tidak output probabilitas | `class_weight='balanced'` |
| **CatBoost** | Gradient boosting khusus fitur kategorikal; tidak butuh encoding | `auto_class_weights='Balanced'` |
| **BalancedRandomForest** | RF + random undersampling per pohon (imbalanced-learn) | Built-in via teknik sampling |

### Catatan Penting
- **Preprocessing identik** dengan notebook utama (SEED=42, split 70/15/15, fitur sama)
- **Threshold optimization** diterapkan ke semua model (kecuali LinearSVC yang tidak output proba → threshold pada decision function)
- **Metrik**: F1 Weighted, F1 Macro, ROC-AUC, Recall (kelas ditangkap), Precision (kelas ditangkap)
- **Tabel perbandingan final** mencakup semua model dari notebook utama + 4 model baru ini

### Struktur Notebook

| Cell | Deskripsi |
|---|---|
| 0 | Header & penjelasan |
| 1 | Install & import library |
| 2 | Load data & preprocessing (identik notebook utama) |
| 3 | Split & encoding (identik notebook utama) |
| 4 | Logistic Regression + threshold optimization |
| 5 | LinearSVC + threshold via decision function |
| 6 | CatBoost + threshold optimization |
| 7 | BalancedRandomForest + threshold optimization |
| 8 | Tabel perbandingan semua 7 model |
| 9 | Visualisasi: ROC curve + F1 Macro bar chart |
| 10 | Analisis: kapan pakai model mana? |

## Cell 1
## Install & Import Library

### Library Tambahan vs Notebook Utama

| Library | Digunakan untuk | Catatan |
|---|---|---|
| `catboost` | Model CatBoost | Perlu install; tidak ada di default Colab |
| `imbalanced-learn` | BalancedRandomForest | Perlu install; extend scikit-learn |
| `sklearn.svm.LinearSVC` | LinearSVC | Sudah ada di scikit-learn |
| `sklearn.linear_model.LogisticRegression` | LR | Sudah ada di scikit-learn |
| `sklearn.preprocessing.StandardScaler` | Scaling untuk LR & SVC | **Wajib** LR dan SVC sensitif terhadap skala |

### Mengapa LR & LinearSVC Butuh StandardScaler tapi LightGBM Tidak?
- **LR**: mengoptimasi koefisien bobot via gradient descent → fitur dengan skala besar (misal `report_delay_days` 0–365) mendominasi gradien
- **LinearSVC**: berbasis jarak margin ke hyperplane → fitur skala besar akan mendominasi perhitungan margin
- **LightGBM / CatBoost / BRF**: berbasis split threshold pohon → scale-invariant, tidak butuh normalisasi

In [ ]:
!pip install catboost imbalanced-learn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings, time, joblib

from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, roc_auc_score, accuracy_score,
    confusion_matrix, classification_report,
    precision_recall_curve, roc_curve
)
from sklearn.dummy import DummyClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV  # untuk LinearSVC → proba
from catboost import CatBoostClassifier
from imblearn.ensemble import BalancedRandomForestClassifier

warnings.filterwarnings('ignore')

SEED = 42

PALETTE = {
    'blue': '#2563EB', 'teal': '#0D9488', 'orange': '#EA580C',
    'red': '#DC2626', 'purple': '#7C3AED', 'gray': '#6B7280',
    'green': '#16A34A', 'yellow': '#CA8A04',
}

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linestyle': '--',
    'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
})

print("Semua library berhasil diimport!")
print(f"scikit-learn, catboost, imbalanced-learn siap digunakan.")

## Cell 2
## Load Data & Preprocessing

### Identik dengan Notebook Utama
Seluruh preprocessing **direplikasi persis** dari notebook utama untuk menjamin komparabilitas:
- SEED = 42
- Kolom yang di-drop identik
- Feature engineering temporal, spasial, dan kategorikal identik
- Missing value handling identik

Ini adalah **syarat mutlak** perbandingan model yang fair jika preprocessing berbeda, perbedaan performa bisa berasal dari data, bukan model.

### Catatan: StandardScaler untuk LR & LinearSVC
StandardScaler di-fit **hanya dari `X_train`**, kemudian di-transform ke `X_val` dan `X_test`.
Ini mencegah **data leakage** informasi distribusi val/test tidak bocor ke proses training.

```python
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)(train)
X_test_sc  = scaler.transform(X_test)
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/Crime_Data')

DATA_PATH = '/content/drive/MyDrive/Crime_Data/archive.zip'

print("Memuat dataset...")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Dimuat: {df_raw.shape[0]:,} baris × {df_raw.shape[1]} kolom")

print("\nMenjalankan preprocessing...")
t0 = time.time()

df = df_raw.copy()

COLS_TO_DROP = ['DR_NO','Rpt Dist No','Crm Cd 2','Crm Cd 3','Crm Cd 4','Cross Street','Mocodes','Crm Cd 1']
df.drop(columns=COLS_TO_DROP, errors='ignore', inplace=True)

df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
df['report_delay_days'] = (df['Date Rptd'] - df['DATE OCC']).dt.days.clip(0, 365)
df.drop(columns=['Date Rptd'], inplace=True)

df['occ_year'] = df['DATE OCC'].dt.year
df['occ_month'] = df['DATE OCC'].dt.month
df['occ_day'] = df['DATE OCC'].dt.day
df['occ_dayofweek'] = df['DATE OCC'].dt.dayofweek
df['occ_quarter'] = df['DATE OCC'].dt.quarter
df['occ_hour'] = (df['TIME OCC'] // 100).clip(0, 23)
df['is_weekend'] = (df['occ_dayofweek'] >= 5).astype(int)
df['is_night'] = ((df['occ_hour'] >= 22) | (df['occ_hour'] < 6)).astype(int)
df['is_rush_hour'] = (df['occ_hour'].between(7,9) | df['occ_hour'].between(16,18)).astype(int)
df['season'] = df['occ_month'].map({12:4,1:4,2:4,3:1,4:1,5:1,6:2,7:2,8:2,9:3,10:3,11:3})
df['time_slot'] = pd.cut(df['occ_hour'], bins=[-1,5,11,17,23], labels=[0,1,2,3]).astype(int)

df['LAT'] = df['LAT'].replace(0, np.nan)
df['LON'] = df['LON'].replace(0, np.nan)
LA_LAT, LA_LON = 34.0522, -118.2437
df['dist_from_center_km'] = np.sqrt((df['LAT']-LA_LAT)**2 + (df['LON']-LA_LON)**2) * 111
valid_mask = df['LAT'].notna() & df['LON'].notna()
km = KMeans(n_clusters=30, random_state=SEED, n_init=10)
geo_labels = km.fit_predict(df.loc[valid_mask, ['LAT','LON']])
df['geo_cluster'] = -1
df.loc[valid_mask, 'geo_cluster'] = geo_labels

df['Weapon Desc'] = df['Weapon Desc'].fillna('NO WEAPON')
df['Weapon Used Cd']= df['Weapon Used Cd'].fillna(0).astype(int)
df['Vict Sex'] = df['Vict Sex'].fillna('X').apply(lambda x: x if x in ['M','F'] else 'X')
df['Vict Descent'] = df['Vict Descent'].fillna('X')
df['Vict Age'] = df['Vict Age'].replace(0, np.nan).clip(upper=100).fillna(df['Vict Age'].median())
df['Premis Desc'] = df['Premis Desc'].fillna(df['Premis Desc'].mode()[0])
df['has_weapon'] = (df['Weapon Desc'] != 'NO WEAPON').astype(int)

def simplify_premis(p):
    p = str(p).upper()
    if any(k in p for k in ['STREET','SIDEWALK','ALLEY','HIGHWAY','ROAD']): return 'RUANG_PUBLIK'
    if any(k in p for k in ['DWELLING','APARTMENT','HOUSE','CONDO','RESIDENCE']): return 'HUNIAN'
    if any(k in p for k in ['PARKING','GARAGE','DRIVEWAY']): return 'PARKIR'
    if any(k in p for k in ['STORE','SHOP','MARKET','MALL','RETAIL']): return 'RITEL'
    if any(k in p for k in ['VEHICLE','AUTO','CAR','TRUCK','BUS']): return 'KENDARAAN'
    if any(k in p for k in ['SCHOOL','COLLEGE','UNIVERSITY']): return 'PENDIDIKAN'
    if any(k in p for k in ['BANK','ATM','FINANCIAL']): return 'KEUANGAN'
    return 'LAINNYA'

def simplify_crime(c):
    c = str(c).upper()
    if any(k in c for k in ['ASSAULT','BATTERY','INTIMATE PARTNER']): return 'KEKERASAN_FISIK'
    if any(k in c for k in ['THEFT','STOLEN','BURGLARY','ROBBERY','SHOPLIFTING']): return 'PENCURIAN_PERAMPOKAN'
    if 'VANDALISM' in c: return 'VANDALISME'
    if any(k in c for k in ['RAPE','SEX','INDECENT','LEWD']): return 'KEJAHATAN_SEKSUAL'
    if any(k in c for k in ['IDENTITY','FRAUD','FORGERY','COUNTERFEIT']): return 'PENIPUAN'
    if any(k in c for k in ['HOMICIDE','MURDER','MANSLAUGHTER']): return 'PEMBUNUHAN'
    if any(k in c for k in ['NARCOTIC','DRUG','MARIJUANA']): return 'NARKOBA'
    if any(k in c for k in ['WEAPON','FIREARM','GUN','KNIFE']): return 'KEPEMILIKAN_SENJATA'
    return 'LAINNYA'

df['premis_category'] = df['Premis Desc'].apply(simplify_premis)
df['crime_category'] = df['Crm Cd Desc'].apply(simplify_crime)

# Label target
df['is_arrested'] = df['Status Desc'].str.contains(
    'Adult Arrest|Juv Arrest', case=False, na=False
).astype(int)

df.drop(columns=['DATE OCC','TIME OCC','Status Desc','Status','Weapon Desc','LAT','LON','Premis Desc','Crm Cd Desc','LOCATION','AREA NAME'], errors='ignore', inplace=True)

print(f"Preprocessing selesai dalam {time.time()-t0:.1f}s")
print(f"Shape akhir: {df.shape}")
print(f"Distribusi target: {df['is_arrested'].value_counts(normalize=True).round(3).to_dict()}")

## Cell 3
## Split & Encoding

### Split Stratified 70/15/15
Identik dengan notebook utama stratify=y menjaga proporsi 9% kelas positif di semua set.

### Dua Versi Data: Raw vs Scaled
Karena notebook ini menggunakan model yang berbeda karakteristiknya:

| Versi | Digunakan untuk | Alasan |
|---|---|---|
| `X_train`, `X_val`, `X_test` | CatBoost, BalancedRF | Tree-based, scale-invariant |
| `X_train_sc`, `X_val_sc`, `X_test_sc` | LogReg, LinearSVC | Distance/gradient-based, butuh scaling |

### Label Encoding
Fitur kategorikal di-encode dengan LabelEncoder agar kompatibel dengan semua model.
**Catatan khusus CatBoost**: CatBoost sebenarnya tidak butuh encoding ia bisa langsung memproses string. Namun untuk konsistensi pipeline, encoding tetap diterapkan dan `cat_features` tidak disebutkan.

In [ ]:
TARGET = 'is_arrested'

FEATURE_COLS = [c for c in df.columns if c != TARGET
                and c not in ['Weapon Used Cd', 'Crm Cd', 'Premis Cd', 'AREA', 'Part 1-2']]

FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

print(f"Jumlah fitur: {len(FEATURE_COLS)}")
print(f"Fitur: {FEATURE_COLS}")

le_dict = {}
df_enc = df.copy()

CAT_COLS = df_enc[FEATURE_COLS].select_dtypes(include='object').columns.tolist()
print(f"\nKolom kategorikal yang di-encode: {CAT_COLS}")

for col in CAT_COLS:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    le_dict[col] = le

X = df_enc[FEATURE_COLS].values
y = df_enc[TARGET].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15/0.85, random_state=SEED, stratify=y_temp
)

print(f"\nUkuran split:")
print(f"Train : {X_train.shape[0]:,} ({y_train.mean():.1%} positif)")
print(f"Val : {X_val.shape[0]:,} ({y_val.mean():.1%} positif)")
print(f"Test : {X_test.shape[0]:,} ({y_test.mean():.1%} positif)")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

print("\nStandardScaler selesai (fit dari train, transform val & test).")

def optimize_threshold(y_val, proba_val, metric='f1_macro'):
    """
    Cari threshold optimal pada val set yang memaksimalkan F1 Macro.
    Return: (best_threshold, best_score)
    """
    thresholds = np.arange(0.20, 0.85, 0.01)
    best_thr, best_score = 0.5, 0

    for thr in thresholds:
        y_pred = (proba_val >= thr).astype(int)
        score = f1_score(y_val, y_pred, average='macro', zero_division=0)
        if score > best_score:
            best_score = score
            best_thr = thr

    return round(best_thr, 2), round(best_score, 4)

def evaluate_model(name, y_test, y_pred, y_proba=None):
    """
    Evaluasi model dan return dict metrik lengkap.
    y_proba opsional jika None, ROC-AUC tidak dihitung.
    """
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    result = {
        'Model' : name,
        'F1 Weighted': round(f1_score(y_test, y_pred, average='weighted'), 4),
        'F1 Macro' : round(f1_score(y_test, y_pred, average='macro'), 4),
        'ROC-AUC' : round(roc_auc_score(y_test, y_proba), 4) if y_proba is not None else None,
        'Recall' : round(tp / (tp + fn) if (tp+fn) > 0 else 0, 4),
        'Precision' : round(tp / (tp + fp) if (tp+fp) > 0 else 0, 4),
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
    }

    print(f"{name}")
    print(f"F1 Weighted : {result['F1 Weighted']}")
    print(f"F1 Macro : {result['F1 Macro']}")
    print(f"ROC-AUC : {result['ROC-AUC']}")
    print(f"Recall : {result['Recall']:.1%}")
    print(f"Precision : {result['Precision']:.1%}")
    print(f"Confusion : TN={tn:,} FP={fp:,} FN={fn:,} TP={tp:,}")

    return result

all_results = {}
all_probas = {}

BASELINE_RESULTS = {
    'LightGBM tuned+thr=0.73': {
        'F1 Weighted': 0.8801, 'F1 Macro': 0.6540, 'ROC-AUC': 0.8324,
        'Recall': 0.43, 'Precision': 0.34
    },
    'Ensemble LGBM+XGB': {
        'F1 Weighted': 0.8801, 'F1 Macro': 0.6540, 'ROC-AUC': 0.8324,
        'Recall': 0.43, 'Precision': 0.34
    },
    'XGBoost': {
        'F1 Weighted': 0.7853, 'F1 Macro': 0.5900, 'ROC-AUC': 0.8286,
        'Recall': 0.78, 'Precision': 0.22
    },
    'Dummy Baseline': {
        'F1 Weighted': 0.8651, 'F1 Macro': 0.4773, 'ROC-AUC': 0.5000,
        'Recall': 0.00, 'Precision': 0.00
    },
}

print("Setup selesai. Siap menjalankan 4 model baru.")

## Cell 4
## Logistic Regression

### Mengapa Logistic Regression sebagai Challenger?
LR adalah **linear baseline terkuat** jika LR sudah bisa mendekati performa gradient boosting, artinya hubungan fitur-target mayoritas bersifat linear dan tidak butuh model kompleks.

### Konfigurasi

| Parameter | Nilai | Alasan |
|---|---|---|
| `solver='saga'` | SGD-based | Paling efisien untuk dataset besar (>100k) dengan regularisasi L1/L2 |
| `penalty='l2'` | Ridge regularisasi | Mencegah overfitting; lebih stabil dari L1 untuk data multi-kolinear |
| `C=1.0` | Inverse regularisasi | Default bisa di-tune via GridSearchCV jika diperlukan |
| `class_weight='balanced'` | Auto-weight | Bobot kelas = N_total / (N_kelas × frekuensi) handling imbalance |
| `max_iter=1000` | Iterasi SGD | Lebih dari default (100) karena dataset besar butuh lebih banyak iterasi |

### Mengapa Butuh Scaling?
LR meminimalkan `log-loss` via gradient descent. Koefisien `β_j` di-update proporsional ke nilai fitur `x_j`. Fitur `report_delay_days` (0–365) akan menghasilkan update gradien 365× lebih besar dari fitur biner (0–1) → StandardScaler menyetarakan kontribusi.

### Threshold Optimization
Sama seperti LightGBM `predict_proba` menghasilkan skor 0–1, threshold optimal dicari dari val set.

In [ ]:
print("Training Logistic Regression...")
t0 = time.time()

# Impute NaNs in scaled data if present
if np.isnan(X_train_sc).any():
    train_medians = np.nanmedian(X_train_sc, axis=0)

    # Use these medians to fill NaNs in X_train_sc, X_val_sc, X_test_sc
    nan_indices_train = np.where(np.isnan(X_train_sc))
    X_train_sc[nan_indices_train] = train_medians[nan_indices_train[1]]

    nan_indices_val = np.where(np.isnan(X_val_sc))
    X_val_sc[nan_indices_val] = train_medians[nan_indices_val[1]]

    nan_indices_test = np.where(np.isnan(X_test_sc))
    X_test_sc[nan_indices_test] = train_medians[nan_indices_test[1]]
    print("NaNs in scaled training, validation, and test sets have been imputed with medians from the training set.")
else:
    print("No NaNs found in scaled training data, no imputation needed.")

lr_model = LogisticRegression(
    solver = 'saga',
    penalty = 'l2',  
    C = 1.0,
    class_weight= 'balanced',
    max_iter = 1000,
    random_state= SEED,
    n_jobs = -1,
)

lr_model.fit(X_train_sc, y_train)
print(f"Training selesai dalam {time.time()-t0:.1f}s")

lr_proba_val = lr_model.predict_proba(X_val_sc)[:, 1]
lr_best_thr, lr_best_f1 = optimize_threshold(y_val, lr_proba_val)

print(f"\nThreshold optimal (val F1 Macro): {lr_best_thr} → F1 Macro val: {lr_best_f1}")

lr_proba_test = lr_model.predict_proba(X_test_sc)[:, 1]
lr_pred_test = (lr_proba_test >= lr_best_thr).astype(int)

lr_result = evaluate_model(
    f'Logistic Regression (thr={lr_best_thr})',
    y_test, lr_pred_test, lr_proba_test
)
all_results['Logistic Regression'] = lr_result
all_probas['Logistic Regression'] = lr_proba_test

coef_df = pd.DataFrame({
    'Fitur' : FEATURE_COLS,
    'Koefisien' : lr_model.coef_[0],
}).sort_values('Koefisien', key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [PALETTE['teal'] if c > 0 else PALETTE['red'] for c in coef_df['Koefisien']]
ax.barh(coef_df['Fitur'], coef_df['Koefisien'], color=colors, edgecolor='white')
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_title(f'Top 15 Koefisien Logistic Regression\n(Teal=positif → dorong ke "ditangkap", Merah=negatif)', fontsize=11)
ax.set_xlabel('Koefisien (setelah StandardScaler)')
plt.tight_layout()
plt.show()

print("\nKoefisien positif terbesar (mendorong ke 'ditangkap'):")
print(coef_df[coef_df['Koefisien'] > 0].head(5).to_string(index=False))
print("\nKoefisien negatif terbesar (mendorong ke 'tidak ditangkap'):")
print(coef_df[coef_df['Koefisien'] < 0].tail(5).to_string(index=False))

## Cell 5
## LinearSVC

### Karakteristik LinearSVC
LinearSVC mencari **hyperplane linear** yang memaksimalkan margin antar kelas. Berbeda dari SVC kernel yang lambat di data besar, LinearSVC menggunakan optimasi **liblinear** sehingga skalabel ke 900k+ sampel.

### Masalah: LinearSVC Tidak Output Probabilitas
`LinearSVC.predict_proba()` tidak ada ia hanya output class label atau **decision function score** (jarak ke hyperplane). Solusi: `CalibratedClassifierCV` membungkus LinearSVC dan mengkalibrasi output jarak ke probabilitas via **Platt scaling**.

```python
# Tanpa kalibrasi → tidak bisa threshold optimization
svc_raw = LinearSVC()

# Dengan kalibrasi → output probabilitas 0–1
svc_cal = CalibratedClassifierCV(LinearSVC(), cv=3, method='sigmoid')
```

### Konfigurasi

| Parameter | Nilai | Alasan |
|---|---|---|
| `C=1.0` | Regularisasi | Trade-off margin vs misclassification |
| `class_weight='balanced'` | Imbalance | Bobot kelas otomatis dari distribusi |
| `max_iter=2000` | Konvergensi | Dataset besar butuh lebih banyak iterasi liblinear |
| `CalibratedClassifierCV(cv=3)` | Platt scaling | Konversi decision score ke probabilitas |

### Ekspektasi Performa
LinearSVC biasanya lebih baik dari LR untuk data dengan banyak fitur kategorikal yang di-encode integer (karena SVM margin lebih robust terhadap outlier). Namun masih kalah dari gradient boosting untuk data dengan interaksi non-linear kuat.

In [ ]:
print("Training LinearSVC (dengan CalibratedClassifierCV untuk probabilitas)...")
print("Estimasi waktu: 3–8 menit untuk 600k+ sampel...")
t0 = time.time()

svc_base = LinearSVC(
    C = 1.0,
    class_weight = 'balanced',
    max_iter = 2000,
    random_state = SEED,
)

svc_model = CalibratedClassifierCV(svc_base, cv=3, method='sigmoid')
svc_model.fit(X_train_sc, y_train)

print(f"Training selesai dalam {time.time()-t0:.1f}s")

svc_proba_val = svc_model.predict_proba(X_val_sc)[:, 1]
svc_best_thr, svc_best_f1 = optimize_threshold(y_val, svc_proba_val)

print(f"Threshold optimal: {svc_best_thr} → val F1 Macro: {svc_best_f1}")

svc_proba_test = svc_model.predict_proba(X_test_sc)[:, 1]
svc_pred_test = (svc_proba_test >= svc_best_thr).astype(int)

svc_result = evaluate_model(
    f'LinearSVC + Platt (thr={svc_best_thr})',
    y_test, svc_pred_test, svc_proba_test
)
all_results['LinearSVC'] = svc_result
all_probas['LinearSVC'] = svc_proba_test

svc_dec_val = svc_base.fit(X_train_sc, y_train).decision_function(X_val_sc)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, color, name in [(0, PALETTE['gray'], 'Tidak Ditangkap'), (1, PALETTE['teal'], 'Ditangkap')]:
    axes[0].hist(svc_dec_val[y_val == label], bins=60, alpha=0.6,
                 color=color, label=name, density=True)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Boundary (score=0)')
axes[0].set_title('Distribusi Decision Score LinearSVC per Kelas')
axes[0].set_xlabel('Decision Function Score')
axes[0].legend()

for label, color, name in [(0, PALETTE['gray'], 'Tidak Ditangkap'), (1, PALETTE['teal'], 'Ditangkap')]:
    axes[1].hist(svc_proba_val[y_val == label], bins=60, alpha=0.6,
                 color=color, label=name, density=True)
axes[1].axvline(svc_best_thr, color='orange', linestyle='--', 
                linewidth=1.5, label=f'Threshold opt ({svc_best_thr})')
axes[1].set_title('Distribusi Probabilitas Setelah Platt Scaling')
axes[1].set_xlabel('Probabilitas P(ditangkap)')
axes[1].legend()

plt.suptitle('LinearSVC: Decision Score vs Calibrated Probability', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Cell 6
## CatBoost

### Mengapa CatBoost Berbeda dari LightGBM/XGBoost?
CatBoost (Categorical Boosting) dirancang khusus untuk dataset dengan banyak **fitur kategorikal**. Perbedaan kunci dari LightGBM:

| Aspek | LightGBM | CatBoost |
|---|---|---|
| Fitur kategorikal | Perlu encoding manual | Built-in, tidak butuh encoding |
| Ordered boosting | Tidak | Ya mencegah target leakage di dalam pohon |
| GPU support | Ya | Ya |
| Kecepatan (CPU) | Sangat cepat (GOSS) | Lebih lambat tapi akurasi sering lebih baik |

Dalam notebook ini, fitur sudah di-encode dengan LabelEncoder untuk konsistensi jadi CatBoost berjalan seperti gradient boosting biasa tanpa manfaat built-in encoding-nya. Namun **ordered boosting** tetap aktif.

### Konfigurasi

| Parameter | Nilai | Alasan |
|---|---|---|
| `iterations=500` | Jumlah pohon | Setara `n_estimators` LightGBM |
| `learning_rate=0.05` | LR | Konsisten dengan hasil Optuna LightGBM |
| `depth=6` | Kedalaman pohon | Setara hasil Optuna LightGBM |
| `auto_class_weights='Balanced'` | Imbalance | Ekuivalen `class_weight='balanced'` |
| `eval_metric='F1'` | Metrik validasi | Optimasi langsung F1 di early stopping |
| `early_stopping_rounds=50` | Stopping | Hentikan jika val F1 tidak membaik |

In [ ]:
print("Training BalancedRandomForest...")
print("Estimasi waktu: 5–12 menit (300 pohon, data besar)...")
t0 = time.time()

brf_model = BalancedRandomForestClassifier(
    n_estimators = 300,
    max_depth = 15,
    sampling_strategy = 'auto', 
    replacement = True, 
    random_state = SEED,
    n_jobs = -1,
    class_weight = None, 
)

brf_model.fit(X_train, y_train)
print(f"Training selesai dalam {time.time()-t0:.1f}s")

brf_proba_val = brf_model.predict_proba(X_val)[:, 1]
brf_best_thr, brf_best_f1 = optimize_threshold(y_val, brf_proba_val)
print(f"Threshold optimal: {brf_best_thr} → val F1 Macro: {brf_best_f1}")

brf_proba_test = brf_model.predict_proba(X_test)[:, 1]
brf_pred_test = (brf_proba_test >= brf_best_thr).astype(int)

brf_result = evaluate_model(
    f'BalancedRandomForest (thr={brf_best_thr})',
    y_test, brf_pred_test, brf_proba_test
)
all_results['BalancedRandomForest'] = brf_result
all_probas['BalancedRandomForest'] = brf_proba_test

brf_fi = pd.DataFrame({
    'Fitur' : FEATURE_COLS,
    'Importance': brf_model.feature_importances_,
}).sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(brf_fi['Fitur'], brf_fi['Importance'],
        color=PALETTE['green'], edgecolor='white')
ax.set_title('BalancedRandomForest Feature Importance (Top 15)')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

print(f"\nJumlah estimator: {len(brf_model.estimators_)}")
print(f"Fitur terpenting: {brf_fi.head(5)['Fitur'].tolist()}")

## Cell 7
## BalancedRandomForest

### Apa itu BalancedRandomForest?
BalancedRandomForest (BRF) dari library `imbalanced-learn` adalah Random Forest yang dimodifikasi khusus untuk class imbalance:

**Random Forest biasa**: setiap pohon dilatih dari bootstrap sample dengan proporsi kelas **sama** dengan dataset original (~9% positif)

**BalancedRandomForest**: setiap pohon dilatih dari bootstrap sample yang **dibalanced** kelas minoritas di-oversample (atau kelas mayoritas di-undersample) sehingga proporsi menjadi 50:50

Ini berbeda dari `class_weight='balanced'` yang hanya mengubah bobot di loss function tanpa mengubah distribusi data yang dilihat tiap pohon.

### Konfigurasi

| Parameter | Nilai | Alasan |
|---|---|---|
| `n_estimators=300` | Jumlah pohon | Lebih banyak dari default (100) untuk stabilitas |
| `max_depth=15` | Kedalaman | Tidak terlalu dalam untuk mencegah overfitting |
| `sampling_strategy='auto'` | Teknik sampling | Undersample mayoritas ke ukuran minoritas per pohon |
| `replacement=True` | Sampling dengan replacement | Bootstrap klasik — lebih stabil dari tanpa replacement |
| `n_jobs=-1` | Paralel | Pakai semua CPU core |

### Ekspektasi
BRF biasanya memberikan **recall tertinggi** karena setiap pohon melihat data seimbang. Namun precision bisa lebih rendah. Trade-off ini bisa dikontrol via threshold optimization.

In [ ]:
rows = []

for model_name, res in BASELINE_RESULTS.items():
    rows.append({
        'Model' : model_name,
        'F1 Weighted': res['F1 Weighted'],
        'F1 Macro' : res['F1 Macro'],
        'ROC-AUC' : res['ROC-AUC'],
        'Recall' : res['Recall'],
        'Precision' : res['Precision'],
        'Sumber' : 'Notebook Utama',
    })

for model_name, res in all_results.items():
    rows.append({
        'Model' : res['Model'],
        'F1 Weighted': res['F1 Weighted'],
        'F1 Macro' : res['F1 Macro'],
        'ROC-AUC' : res['ROC-AUC'],
        'Recall' : res['Recall'],
        'Precision' : res['Precision'],
        'Sumber' : 'Notebook Ini',
    })

comparison_df = pd.DataFrame(rows).sort_values('F1 Macro', ascending=False)
comparison_df = comparison_df.reset_index(drop=True)
comparison_df.index += 1

print(f"{'RANKING':<8} {'MODEL':<40} {'F1W':>7} {'F1M':>7} {'AUC':>7} {'RECALL':>8} {'PREC':>7}")

for i, row in comparison_df.iterrows():
    tag = '★ TERBAIK' if i == 1 else ('← BASELINE' if row['Model']=='Dummy Baseline' else '')
    auc = f"{row['ROC-AUC']:.4f}" if row['ROC-AUC'] else "N/A"
    print(f"#{i:<7} {row['Model']:<40} {row['F1 Weighted']:>7.4f} {row['F1 Macro']:>7.4f} "
          f"{auc:>7} {row['Recall']:>8.2%} {row['Precision']:>7.2%}  {tag}")

comparison_df.to_csv('/content/drive/MyDrive/Crime_Data/full_model_comparison.csv', index=False)
print("\nTabel perbandingan disimpan ke Google Drive.")

## Cell 8
## Tabel Perbandingan Semua Model

### Cara Baca Tabel
- **F1 Macro** adalah metrik utama tidak tertipu imbalance, bobot kelas setara
- **Recall** kelas ditangkap seberapa banyak kasus penangkapan yang berhasil terdeteksi
- **Precision** kelas ditangkap dari prediksi "ditangkap", berapa yang benar
- **ROC-AUC** kemampuan diskriminasi di semua threshold, tidak bergantung threshold spesifik

### Interpretasi Trade-off Recall vs Precision
Threshold yang lebih rendah → recall naik, precision turun (lebih banyak alarm, lebih banyak false alarm)
Threshold yang lebih tinggi → precision naik, recall turun (lebih selektif, tapi lebih banyak yang terlewat)

Untuk konteks LAPD, **recall lebih penting** (jangan biarkan kasus yang bisa diselesaikan terlewat), tapi precision juga perlu cukup tinggi (tidak buang sumber daya ke false alarm).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
colors_map = {
    'Logistic Regression' : PALETTE['blue'],
    'LinearSVC' : PALETTE['orange'],
    'CatBoost' : PALETTE['purple'],
    'BalancedRandomForest': PALETTE['green'],
}

for model_name, proba in all_probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.4f})',
            color=colors_map.get(model_name, PALETTE['gray']), linewidth=2)

ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Random Baseline (AUC=0.500)')
ax.axhline(y=0.43, color=PALETTE['teal'], linestyle=':', linewidth=1,
           label='LightGBM Tuned (TPR=43% @ thr=0.73, AUC=0.832)')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve — 4 Model Baru vs LightGBM Referensi')
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
all_models_for_bar = comparison_df.head(8)
bar_colors = []
for _, row in all_models_for_bar.iterrows():
    if row['Sumber'] == 'Notebook Ini':
        bar_colors.append(PALETTE['purple'])
    elif 'LightGBM tuned' in str(row['Model']):
        bar_colors.append(PALETTE['teal'])
    elif 'Dummy' in str(row['Model']):
        bar_colors.append(PALETTE['gray'])
    else:
        bar_colors.append(PALETTE['blue'])

bars = ax2.bar(range(len(all_models_for_bar)), all_models_for_bar['F1 Macro'],
               color=bar_colors, edgecolor='white', width=0.6)
ax2.set_xticks(range(len(all_models_for_bar)))
ax2.set_xticklabels([m[:25] for m in all_models_for_bar['Model']],
                    rotation=35, ha='right', fontsize=8)
ax2.set_title('F1 Macro Semua Model (Ungu=Model Baru, Teal=LightGBM Terbaik)')
ax2.set_ylabel('F1 Macro')
ax2.axhline(y=0.654, color=PALETTE['teal'], linestyle='--',
            linewidth=1.5, label='LightGBM Tuned (0.654)')
ax2.legend(fontsize=8)
ax2.set_ylim(0, 0.8)

for bar, val in zip(bars, all_models_for_bar['F1 Macro']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))

scatter_models = comparison_df[comparison_df['Model'] != 'Dummy Baseline']
scatter_colors = [PALETTE['purple'] if s=='Notebook Ini' else PALETTE['teal']
                  for s in scatter_models['Sumber']]

ax.scatter(scatter_models['Recall'], scatter_models['Precision'],
           c=scatter_colors, s=180, edgecolors='white', linewidth=1.5, zorder=5)

for _, row in scatter_models.iterrows():
    ax.annotate(row['Model'][:22],
                (row['Recall'], row['Precision']),
                textcoords='offset points', xytext=(8, 4),
                fontsize=8, color='white')

ax.set_xlabel('Recall (kelas ditangkap) seberapa banyak yang terdeteksi')
ax.set_ylabel('Precision (kelas ditangkap) dari prediksi ditangkap, berapa yang benar')
ax.set_title('Trade-off Recall vs Precision per Model\n(Ungu=Model Baru, Teal=Notebook Utama)')
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 0.6)

plt.tight_layout()
plt.show()

## Cell 9
## Visualisasi: ROC Curve + Bar Chart Perbandingan

### Panel Visualisasi
1. **ROC Curve semua model** perbandingan AUC di semua threshold
2. **F1 Macro bar chart** perbandingan metrik utama per model
3. **Radar chart** profil multidimensi tiap model (F1W, F1M, AUC, Recall, Precision)
4. **Scatter: Recall vs Precision** trade-off tiap model

### Membaca Radar Chart
Setiap sumbu mewakili satu metrik (dinormalisasi 0–1). Model ideal mengisi seluruh area radar. Model yang "cramped" ke beberapa sumbu menunjukkan trade-off yang jelas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
colors_map = {
    'Logistic Regression' : PALETTE['blue'],
    'LinearSVC' : PALETTE['orange'],
    'CatBoost' : PALETTE['purple'],
    'BalancedRandomForest': PALETTE['green'],
}

for model_name, proba in all_probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.4f})',
            color=colors_map.get(model_name, PALETTE['gray']), linewidth=2)

ax.plot([0,1],[0,1], 'k--', linewidth=1, label='Random Baseline (AUC=0.500)')
ax.axhline(y=0.43, color=PALETTE['teal'], linestyle=':', linewidth=1,
           label='LightGBM Tuned (TPR=43% @ thr=0.73, AUC=0.832)')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve — 4 Model Baru vs LightGBM Referensi')
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
all_models_for_bar = comparison_df.head(8)
bar_colors = []
for _, row in all_models_for_bar.iterrows():
    if row['Sumber'] == 'Notebook Ini':
        bar_colors.append(PALETTE['purple'])
    elif 'LightGBM tuned' in str(row['Model']):
        bar_colors.append(PALETTE['teal'])
    elif 'Dummy' in str(row['Model']):
        bar_colors.append(PALETTE['gray'])
    else:
        bar_colors.append(PALETTE['blue'])

bars = ax2.bar(range(len(all_models_for_bar)), all_models_for_bar['F1 Macro'],
               color=bar_colors, edgecolor='white', width=0.6)
ax2.set_xticks(range(len(all_models_for_bar)))
ax2.set_xticklabels([m[:25] for m in all_models_for_bar['Model']],
                    rotation=35, ha='right', fontsize=8)
ax2.set_title('F1 Macro Semua Model (Ungu=Model Baru, Teal=LightGBM Terbaik)')
ax2.set_ylabel('F1 Macro')
ax2.axhline(y=0.654, color=PALETTE['teal'], linestyle='--',
            linewidth=1.5, label='LightGBM Tuned (0.654)')
ax2.legend(fontsize=8)
ax2.set_ylim(0, 0.8)

for bar, val in zip(bars, all_models_for_bar['F1 Macro']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))

scatter_models = comparison_df[comparison_df['Model'] != 'Dummy Baseline']
scatter_colors = [PALETTE['purple'] if s=='Notebook Ini' else PALETTE['teal']
                  for s in scatter_models['Sumber']]

ax.scatter(scatter_models['Recall'], scatter_models['Precision'],
           c=scatter_colors, s=180, edgecolors='white', linewidth=1.5, zorder=5)

for _, row in scatter_models.iterrows():
    ax.annotate(row['Model'][:22],
                (row['Recall'], row['Precision']),
                textcoords='offset points', xytext=(8, 4),
                fontsize=8, color='white')

ax.set_xlabel('Recall (kelas ditangkap) seberapa banyak yang terdeteksi')
ax.set_ylabel('Precision (kelas ditangkap) dari prediksi ditangkap, berapa yang benar')
ax.set_title('Trade-off Recall vs Precision per Model\n(Ungu=Model Baru, Teal=Notebook Utama)')
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 0.6)

plt.tight_layout()
plt.show()

## Cell 10
## Analisis: Kapan Pakai Model Mana?

### Ringkasan Karakteristik Tiap Model

| Model | Kecepatan Training | Interpretabilitas | Keunggulan | Kelemahan |
|---|---|---|---|---|
| **LightGBM** | Sangat cepat | Sedang (SHAP) | Performa terbaik overall | Black box |
| **CatBoost** | Lambat | Sedang | Stabil, fitur kategorikal | Butuh waktu lebih |
| **BalancedRF** | Sedang | Sedang | Recall tinggi, robust | Precision rendah |
| **Logistic Regression** | Sangat cepat | Sangat tinggi | Koefisien langsung interpretabel | Hanya tangkap hubungan linear |
| **LinearSVC** | Cepat | Tinggi | Robust noise, skalabel | Tidak output probabilitas native |

### Rekomendasi Penggunaan

**Jika performa adalah prioritas utama** → tetap pakai LightGBM tuned + threshold 0.73

**Jika interpretabilitas dibutuhkan** (untuk laporan ke pejabat non-teknis) → Logistic Regression, karena koefisien bisa langsung dibaca sebagai "fitur X meningkatkan probabilitas penangkapan sebesar Y%"

**Jika recall adalah prioritas** (tidak ingin melewatkan kasus yang bisa diselesaikan) → BalancedRandomForest, karena built-in oversampling memaksimalkan deteksi kelas minoritas

**Jika data baru terus masuk dan butuh update cepat** → Logistic Regression atau LinearSVC, karena training ulang jauh lebih cepat dari gradient boosting

**Jika ada banyak fitur kategorikal baru** → CatBoost, karena tidak perlu re-encoding manual


In [ ]:
print("Training CatBoostClassifier...")
print("Estimasi waktu: 3-8 menit...")
t0_cat = time.time()

cat_features_indices = [FEATURE_COLS.index(col) for col in CAT_COLS if col in FEATURE_COLS]

from catboost import Pool

X_train_df_cb = pd.DataFrame(X_train, columns=FEATURE_COLS)
X_val_df_cb = pd.DataFrame(X_val, columns=FEATURE_COLS)
X_test_df_cb = pd.DataFrame(X_test, columns=FEATURE_COLS)

for col_name in CAT_COLS:
    X_train_df_cb[col_name] = X_train_df_cb[col_name].astype(int)
    X_val_df_cb[col_name] = X_val_df_cb[col_name].astype(int)
    X_test_df_cb[col_name] = X_test_df_cb[col_name].astype(int)

train_pool = Pool(X_train_df_cb, y_train, cat_features=cat_features_indices)
val_pool = Pool(X_val_df_cb, y_val, cat_features=cat_features_indices)
test_pool = Pool(X_test_df_cb, cat_features=cat_features_indices)

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    loss_function='Logloss',
    eval_metric='F1',
    random_seed=SEED,
    verbose=0,
    early_stopping_rounds=50,
)

cat_model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50, verbose=False)

print(f"Training selesai dalam {time.time()-t0_cat:.1f}s")

cat_proba_val = cat_model.predict_proba(val_pool)[:, 1]
cat_best_thr, cat_best_f1 = optimize_threshold(y_val, cat_proba_val)
print(f"Threshold optimal: {cat_best_thr} → val F1 Macro: {cat_best_f1}")

cat_proba_test = cat_model.predict_proba(test_pool)[:, 1]
cat_pred_test = (cat_proba_test >= cat_best_thr).astype(int)

cat_result = evaluate_model(
    f'CatBoost (thr={cat_best_thr})',
    y_test, cat_pred_test, cat_proba_test
)
all_results['CatBoost'] = cat_result
all_probas['CatBoost'] = cat_proba_test

cat_fi = pd.DataFrame({
    'Fitur': FEATURE_COLS,
    'Importance': cat_model.get_feature_importance(train_pool), # Pass train_pool here
}).sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(cat_fi['Fitur'], cat_fi['Importance'],
        color=PALETTE['purple'], edgecolor='white')
ax.set_title('CatBoost Feature Importance (Top 15)')
ax.set_xlabel('Feature Importance')
plt.tight_layout()
plt.show()

print(f"\nFitur terpenting: {cat_fi.head(5)['Fitur'].tolist()}")

print("RINGKASAN EKSEKUTIF PERBANDINGAN 7 MODEL")

best_f1m = comparison_df.iloc[0]
best_recall = comparison_df.sort_values('Recall', ascending=False).iloc[0]
best_prec = comparison_df.sort_values('Precision', ascending=False).iloc[0]
fastest_model = 'Logistic Regression'

print(f"\n★  F1 Macro tertinggi : {best_f1m['Model']} ({best_f1m['F1 Macro']:.4f})")
print(f"★  Recall tertinggi : {best_recall['Model']} ({best_recall['Recall']:.2%})")
print(f"★  Precision tertinggi : {best_prec['Model']} ({best_prec['Precision']:.2%})")
print(f"★  Paling interpretabel  : Logistic Regression (koefisien langsung terbaca)")

print("\n APAKAH MODEL BARU MENGALAHKAN LIGHTGBM?")
lgbm_f1m = BASELINE_RESULTS['LightGBM tuned+thr=0.73']['F1 Macro']
lgbm_auc = BASELINE_RESULTS['LightGBM tuned+thr=0.73']['ROC-AUC']

for model_name, res in all_results.items():
    beat_f1 = "KALAHKAN" if res['F1 Macro'] > lgbm_f1m else "Di bawah"
    beat_auc = "KALAHKAN" if (res['ROC-AUC'] or 0) > lgbm_auc else "Di bawah"
    print(f"{model_name:<25} F1 Macro: {beat_f1} ({res['F1 Macro']:.4f} vs {lgbm_f1m})  "
          f"AUC: {beat_auc} ({res['ROC-AUC']} vs {lgbm_auc})")

print("\nSIMPAN TABEL FINAL")
comparison_df.to_csv('/content/drive/MyDrive/Crime_Data/full_model_comparison.csv', index=False)
print("Disimpan ke: /content/drive/MyDrive/Crime_Data/full_model_comparison.csv")

print("\nSIMPAN MODEL TERBAIK DARI NOTEBOOK INI")
best_new_model_name = max(all_results, key=lambda k: all_results[k]['F1 Macro'])
print(f"Model terbaik dari notebook ini: {best_new_model_name}")

model_map = {
    'Logistic Regression' : lr_model,
    'LinearSVC' : svc_model,
    'CatBoost' : cat_model,
    'BalancedRandomForest': brf_model,
}

joblib.dump(model_map[best_new_model_name],
            f'/content/drive/MyDrive/Crime_Data/{best_new_model_name.lower().replace(" ","_")}.pkl')
print(f"Model disimpan ke Google Drive.")